# Consolidated research notebook

Method group **G05**: Transformer fine tuning. Architecture: CNN-Transformer encoder. Method tags: fine tuning|attention.

This copy preserves the research implementation while removing saved outputs, execution counters, pinned GPU selection, and scratch-only cells. Closely related variants are summarized in the research notebook index.

In [ ]:
from battery_soh.data import resolve_data_dir
from battery_soh.evaluation import MAPE, RMSPE
from battery_soh.paths import artifact_path
from battery_soh.preprocessing import interpolate_timeseries
from battery_soh.raw_data import load_battery_dictionary
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import random
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
# Shared, portable raw-data loading. This may require substantial memory.
RESEARCH_BATCHES = ("b1", "b2", "b3")
bat_dict = load_battery_dictionary(batches=RESEARCH_BATCHES)


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
batch_keys = [*bat_dict.keys()]
print(len(batch_keys))

In [ ]:
random.seed(12)

In [ ]:
batch_keys = [*bat_dict.keys()]
selected_keys = []

for battery_key in batch_keys:
    if (len(bat_dict[battery_key]['summary']['cycle']) > 250) & (len(bat_dict[battery_key]['summary']['cycle']) < 1500):
        for k in range(1, len(bat_dict[battery_key]['summary']['QD'])):
            QD = bat_dict[battery_key]['summary']['QD'][k]
            temp_SOH = (QD/1.1) * 100
            if temp_SOH < 80.2:
                selected_keys.append(battery_key)
                break

In [ ]:
selected_keys.remove('b2c9')
selected_keys.remove('b2c1')
selected_keys.remove('b2c4')
selected_keys.remove('b2c17')
selected_keys.remove('b2c25')
selected_keys.remove('b2c33')
selected_keys.remove('b2c41')
selected_keys.remove('b2c6')
selected_keys.remove('b2c3')
selected_keys.remove('b2c2')
selected_keys.remove('b2c24')
selected_keys.remove('b3c46')
selected_keys.remove('b3c18')

In [ ]:
random.shuffle(selected_keys)
train_bat_key = selected_keys[:65]
val_bat_key = selected_keys[65:85]
test_bat_key = selected_keys[85:]

In [ ]:
time_value = 5
inter_value = 30

In [ ]:
whole_data =[]
Cycle_true = []
SOH_true = []
Cycle_index = 0

for battery_key in train_bat_key:

    for cycle_num in range(1, len(bat_dict[battery_key]['summary']['cycle'])):
        temporal_data = []
        cycle_num = str(cycle_num)

        data_length = len(bat_dict[battery_key]['cycles'][cycle_num]['Qc'])

        if cycle_num != '1':
            for i in range(11,data_length):
                Charge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qc'][i] / 1.1) * 100
                if Charge_SOC > 80.1:

                    for p in range(11,data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    Distinguish_data = bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx]
                    has_negative = np.any(Distinguish_data < 0)
                    if has_negative:
                        break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Current_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx], 3)
                    Voltage_data = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                    'Current' : Current_data,
                                    'Voltage' : Voltage_data})
                    
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)
                    
                    Current_data = interpolate_timeseries(data[1], inter_value)
                    Voltage_data = interpolate_timeseries(data[2], inter_value)
                    Voltage_data = (Voltage_data - 3.37)/(3.62-3.37)

                    temporal_data.append(Current_data)
                    temporal_data.append(Voltage_data)
                    
                    #if (end_time - start_time) < 5.1:
                        #print(battery_key + ' '+ str(cycle_num))
                    #print('Charge_time : ' + str(end_time - start_time))
                    break

            for i in range(11,len(bat_dict[battery_key]['cycles'][cycle_num]['Qd'])):
                Discharge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qd'][i] / 1.1) * 100
                if Discharge_SOC > 0.01:

                    for p in range(data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Discharge_Voltage = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                    'Voltage' : Discharge_Voltage})
                    
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)

                    Discharge_Voltage = interpolate_timeseries(data[1], inter_value)
                    Discharge_Voltage = (Discharge_Voltage-3)/(3.6-3)


                    temporal_data.append(Discharge_Voltage)


                    break
                    
            if np.array(temporal_data).shape == (3, inter_value):
                Cycle_true.append(bat_dict[battery_key]['summary']['cycle'][int(cycle_num)])
                SOH_true.append(bat_dict[battery_key]['summary']['QD'][int(cycle_num)])
                whole_data.append(np.array(temporal_data))
    
    temporal_Cycle = np.array(Cycle_true)[Cycle_index:]
    max_Cycle = np.max(temporal_Cycle)
    temporal_Cycle = temporal_Cycle/max_Cycle
    temporal_Cycle.tolist()
    
    Cycle_true[Cycle_index:] = temporal_Cycle
    Cycle_index = len(Cycle_true)
    

whole_data = np.array(whole_data)
Cycle_true = np.array(Cycle_true)
SOH_true = np.array(SOH_true)

whole_data.shape

In [ ]:
whole_data = np.reshape(whole_data, (whole_data.shape[0], 1 , whole_data.shape[1], whole_data.shape[2]))
print(whole_data.shape)

In [ ]:
class MyDataset(Dataset):
    def __init__(self, X1,  y1):
        self.X1 = X1
        self.y1 = y1
    
    def __len__(self):
        return len(self.X1)
    
    def __getitem__(self, idx):
        return self.X1[idx], self.y1[idx]
    
dataset = MyDataset(torch.tensor(whole_data, dtype=torch.float32),torch.tensor(SOH_true, dtype=torch.float32))

data_loader = DataLoader(dataset, batch_size=2048, shuffle=True)

In [ ]:
class SOH_model(nn.Module):
    def __init__(self, in_features, time_length, first_head, first_layer):
        super().__init__()
        self.in_features = in_features
        self.time_length = time_length
        self.first_head = first_head
        self.first_layer = first_layer

        self.linear_first = nn.Linear(self.time_length, 100)

        self.conv_1 = nn.Conv2d(1, 10, kernel_size=(1, 10), padding = 'same')
        self.conv_2 = nn.Conv2d(10, 10, kernel_size=(1, 10), padding = 'same')
        self.conv_3 = nn.Conv2d(10, 20, kernel_size=(1, 10), padding = 'same')
        self.conv_4 = nn.Conv2d(20, 10, kernel_size=(1, 5), padding = 'same')
        self.conv_5 = nn.Conv2d(10, 1, kernel_size=(1, 5), padding = 'same')
        self.conv_6 = nn.Conv2d(1, 1, kernel_size=(self.in_features, self.in_features)
                                , padding = 'same')

        self.encoder_layer = nn.TransformerEncoderLayer(d_model=100, nhead=self.first_head, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer, num_layers=self.first_layer)

        self.linear1 = nn.Linear(100 * self.in_features, 50)
        self.linear2 = nn.Linear(50, 1)

    def forward(self, x):
        x = self.linear_first(x)
        x1 = x
        # MSDCNN
        x = self.conv_1(x)
        x = F.leaky_relu(x)
        x = self.conv_2(x)
        x = F.leaky_relu(x)
        x = self.conv_3(x)
        x = F.leaky_relu(x)
        x = self.conv_4(x)
        x = F.leaky_relu(x)
        x = self.conv_5(x)
        x = F.leaky_relu(x)
        x = x + x1 # Skip connection
        x = self.conv_6(x)
        x = F.leaky_relu(x)
        
        x = x.squeeze(1)
        
        # Multi Head Attention
        x = self.encoder(x)
        x = x.reshape(-1, 100 * self.in_features)
        x = self.linear1(x)
        x = F.leaky_relu(x)
        x = self.linear2(x)

        return x.squeeze(1)

In [ ]:
whole_val_data =[]
Cycle_val = []
SOH_val = []
Cycle_index = 0

for battery_key in val_bat_key:
    for cycle_num in range(2, len(bat_dict[battery_key]['summary']['cycle'])):
        temporal_data = []
        cycle_num = str(cycle_num)

        data_length = len(bat_dict[battery_key]['cycles'][cycle_num]['Qc'])

        if cycle_num != '1':
            for i in range(11,data_length):
                Charge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qc'][i] / 1.1) * 100
                if Charge_SOC > 80.1:

                    for p in range(11,data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    Distinguish_data = bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx]
                    has_negative = np.any(Distinguish_data < 0)
                    if has_negative:
                        break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Current_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx], 3)
                    Voltage_data = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                    'Current' : Current_data,
                                    'Voltage' : Voltage_data})
                    
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)

                    Current_data = interpolate_timeseries(data[1], inter_value)
                    Voltage_data = interpolate_timeseries(data[2], inter_value)
                    
                    Voltage_data = (Voltage_data - 3.37)/(3.62-3.37)


                    temporal_data.append(Current_data)
                    temporal_data.append(Voltage_data)
                    
                    break

            for i in range(11,len(bat_dict[battery_key]['cycles'][cycle_num]['Qd'])):
                Discharge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qd'][i] / 1.1) * 100
                if Discharge_SOC > 0.01:

                    for p in range(data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Discharge_Voltage = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                    'Voltage' : Discharge_Voltage})
                    
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)
                    Discharge_Voltage = interpolate_timeseries(data[1], inter_value)
                    Discharge_Voltage = (Discharge_Voltage-3)/(3.6-3)

                    temporal_data.append(Discharge_Voltage)


                    break
            if np.array(temporal_data).shape == (3, inter_value):
                Cycle_val.append(bat_dict[battery_key]['summary']['cycle'][int(cycle_num)])
                SOH_val.append(bat_dict[battery_key]['summary']['QD'][int(cycle_num)])
                whole_val_data.append(np.array(temporal_data))
    
    temporal_Cycle = np.array(Cycle_val)[Cycle_index:]
    max_Cycle = np.max(temporal_Cycle)
    temporal_Cycle = temporal_Cycle/max_Cycle
    temporal_Cycle.tolist()
    
    Cycle_val[Cycle_index:] = temporal_Cycle
    Cycle_index = len(Cycle_val)

whole_val_data = np.array(whole_val_data)
Cycle_val = np.array(Cycle_val)
SOH_val = np.array(SOH_val)

whole_val_data = np.reshape(whole_val_data, (whole_val_data.shape[0], 1, whole_val_data.shape[1], whole_val_data.shape[2]))

whole_val_data = torch.tensor(whole_val_data, dtype=torch.float32)

In [ ]:
print(whole_val_data.shape)
whole_val_data = whole_val_data.to(DEVICE)

In [ ]:
model = SOH_model(3, inter_value, 4,2).to(DEVICE)
model = model.train()

optimizer = torch.optim.Adam(model.parameters())
loss_fn = torch.nn.MSELoss()

SOH_loss_best = 1000


for epoch in range(0,500):
    idx = 0
    for batch_X1, batch_y1 in data_loader:
        model.train()
        batch_X1 = batch_X1.to(DEVICE)
        batch_y1 = batch_y1.to(DEVICE)
        SOH_pred= model(batch_X1)
        loss = torch.sqrt(loss_fn(SOH_pred, batch_y1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        model.eval()
        SOH_pred_val = model(whole_val_data)
        SOH_pred_val = SOH_pred_val.cpu().detach().numpy()
        SOH_val_loss = mean_squared_error(SOH_val, SOH_pred_val)**0.5
        


        if SOH_val_loss < SOH_loss_best:
            SOH_loss_best = SOH_val_loss
            torch.save(model.state_dict(), artifact_path("New_normalization_300_fine_tune"))
            print('epoch : ', epoch ,'/idx : ', idx , '/train_loss :', round(float(loss),4), '/SOH_val_loss : ', round(SOH_val_loss, 4))
        
        idx += 1

In [ ]:
rmse_list = []
mape_list = []

mode = 'SOH'

fig = plt.figure(figsize = (20,12))

for k in range(len(test_bat_key)-10):   
#for k in range(10,len(test_bat_key)):   
    battery_list = [test_bat_key[k]]

    model_new = SOH_model(3, inter_value, 4,2).to(DEVICE)
    
    model_new.load_state_dict(torch.load(artifact_path("New_normalization_300_fine_tune")))
    model_new.eval()

    whole_test_data =[]
    Cycle_test = []
    SOH_test = []
    Cycle_index = 0

    for battery_key in battery_list:
        for cycle_num in range(1, len(bat_dict[battery_key]['summary']['cycle'])):
            temporal_data = []
            cycle_num = str(cycle_num)

            data_length = len(bat_dict[battery_key]['cycles'][cycle_num]['Qc'])

            if cycle_num != '1':
                for i in range(11,data_length):
                    Charge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qc'][i] / 1.1) * 100
                    if Charge_SOC > 80.1:

                        for p in range(11,data_length):
                            start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                            end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                            total_time = end_time -start_time
                            if total_time > time_value:
                                final_idx = p - 1
                                break

                        Distinguish_data = bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx]
                        has_negative = np.any(Distinguish_data < 0)
                        if has_negative:
                            break

                        time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                        Current_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx], 3)
                        Voltage_data = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                        df = pd.DataFrame({'time' : time_data,
                                        'Current' : Current_data,
                                        'Voltage' : Voltage_data})
                        
                        df = df.drop_duplicates(['time'])

                        data = np.array(df.T)

                        Current_data = interpolate_timeseries(data[1], inter_value)
                        Voltage_data = interpolate_timeseries(data[2], inter_value)
                        
                        Voltage_data = (Voltage_data - 3.37)/(3.62-3.37)

                        temporal_data.append(Current_data)
                        temporal_data.append(Voltage_data)
                        
                        #if (end_time - start_time) < 5.1:
                            #print(battery_key + ' '+ str(cycle_num))
                        #print('Charge_time : ' + str(end_time - start_time))
                        break

                for i in range(11,len(bat_dict[battery_key]['cycles'][cycle_num]['Qd'])):
                    Discharge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qd'][i] / 1.1) * 100
                    if Discharge_SOC > 0.01:

                        for p in range(data_length):
                            start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                            end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                            total_time = end_time -start_time
                            if total_time > time_value:
                                final_idx = p - 1
                                break

                        time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                        Discharge_Voltage = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                        df = pd.DataFrame({'time' : time_data,
                                        'Voltage' : Discharge_Voltage})
                        
                        df = df.drop_duplicates(['time'])

                        data = np.array(df.T)
                        Discharge_Voltage = interpolate_timeseries(data[1], inter_value)
                        Discharge_Voltage = (Discharge_Voltage-3)/(3.6-3)

                        temporal_data.append(Discharge_Voltage)


                        break
                if np.array(temporal_data).shape == (3, inter_value):
                    Cycle_test.append(bat_dict[battery_key]['summary']['cycle'][int(cycle_num)])
                    SOH_test.append(bat_dict[battery_key]['summary']['QD'][int(cycle_num)])
                    whole_test_data.append(np.array(temporal_data))

        temporal_Cycle = np.array(Cycle_test)[Cycle_index:]
        max_Cycle = np.max(temporal_Cycle)
        temporal_Cycle = temporal_Cycle/max_Cycle
        temporal_Cycle.tolist()

        Cycle_test[Cycle_index:] = temporal_Cycle
        Cycle_index = len(Cycle_test)

    whole_test_data = np.array(whole_test_data)
    Cycle_test = np.array(Cycle_test)
    SOH_test = np.array(SOH_test)

    whole_test_data = np.reshape(whole_test_data, (whole_test_data.shape[0], 1, whole_test_data.shape[1], whole_test_data.shape[2]))

    whole_test_data = torch.tensor(whole_test_data, dtype=torch.float32)


    model_new.eval()

    whole_test_data = whole_test_data.to(DEVICE)
    test_data = test_data.to(DEVICE)

    SOH_test_pred = model_new(whole_test_data)
    SOH_test_pred = SOH_test_pred.cpu().detach().numpy()
    SOH_test= np.array(SOH_test)
    
    if battery_key == 'b2c14':
        SOH_test_pred = np.delete(SOH_test_pred, [246])
        Cycle_test = np.delete(Cycle_test, [246])
        SOH_test = np.delete(SOH_test, [246])
    

    plot_num = 331 + k
    #plot_num = 331 + (k-10)
    
    if mode == 'SOH':
        
        
        soh_rmse = mean_squared_error(SOH_test, SOH_test_pred )**0.5
        soh_mape = MAPE(np.array(SOH_test), np.array(SOH_test_pred ))
        
        rmse_list.append(soh_rmse)
        mape_list.append(soh_mape)

        plt.subplot(plot_num)
        plt.plot(SOH_test_pred , color = 'r', label = 'Estimated SOH')
        plt.plot(SOH_test, color = 'b', label = 'real SOH')
        plt.text(1, 0.95, 'RMSE : ' + str(round(soh_rmse, 5)), fontsize = 20)
        plt.text(1, 0.93, 'MAPE(%) : ' + str(round(soh_mape, 5)), fontsize = 20)

        plt.title(battery_list[0], fontsize = 15)
        plt.ylabel('Capacity (AH)' , fontsize=15)
        plt.xlabel('cycle', fontsize=15)
        plt.legend()

    
fig.tight_layout(pad=3.0)
plt.show()

In [ ]:
plt.plot(SOH_test_pred[247:])

In [ ]:
fig, ax = plt.subplots()

rmse_arr = np.array(rmse_list)*100/1.1
rmse_avg = np.mean(rmse_arr)

mape_arr = np.array(mape_list)
mape_avg = np.mean(mape_arr)

ax.boxplot([rmse_arr, mape_arr])
plt.xticks([1, 2],['RMSE(%)', 'MAPE'])
plt.text(0.6, 0.5, '[RMSE_AVG]\n   ' + str(round(rmse_avg, 3)) + '%')
plt.text(1.6, 0.5, '[MAPE_AVG]\n   ' + str(round(mape_avg, 3)) + '%')

plt.show

In [ ]:
model_new = SOH_model(4, inter_value, 2, 2, 2, 7).to(DEVICE)

model_new.register_parameter('par_a', par_a)
model_new.register_parameter('par_b', par_b)
model_new.register_parameter('par_c', par_c)
model_new.register_parameter('par_d', par_d)
    
model_new.register_parameter('lam_1', lam_1)
model_new.register_parameter('lam_2', lam_2)
model_new.register_parameter('lam_3', lam_3)
#model_new.register_parameter('lam_4', lam_4)
    
model_new.load_state_dict(torch.load(artifact_path("New_normalization_180_SOH_1")))
model_new.eval()

In [ ]:
whole_val_data =[]
Cycle_val = []
SOH_val = []
Cycle_index = 0

for battery_key in test_bat_key:
    for cycle_num in range(2, len(bat_dict[battery_key]['summary']['cycle'])):
        temporal_data = []
        cycle_num = str(cycle_num)

        data_length = len(bat_dict[battery_key]['cycles'][cycle_num]['Qc'])

        if cycle_num != '1':
            for i in range(11,data_length):
                Charge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qc'][i] / 1.1) * 100
                if Charge_SOC > 80.1:

                    for p in range(11,data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    Distinguish_data = bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx]
                    has_negative = np.any(Distinguish_data < 0)
                    if has_negative:
                        break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Current_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx], 3)
                    Voltage_data = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]
                    Temperature_data = bat_dict[battery_key]['cycles'][cycle_num]['T'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                    'Current' : Current_data,
                                    'Voltage' : Voltage_data})
                    
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)

                    Current_data = interpolate_timeseries(data[1], inter_value)
                    Voltage_data = interpolate_timeseries(data[2], inter_value)
                    Temperature_data = interpolate_timeseries(Temperature_data, inter_value)
                    
                    Voltage_data = (Voltage_data - 3.37)/(3.62-3.37)
                    Temperature_data = (Temperature_data - 27.2)/(41-27.2)


                    temporal_data.append(Current_data)
                    temporal_data.append(Voltage_data)
                    temporal_data.append(Temperature_data)
                    
                    break

            for i in range(11,len(bat_dict[battery_key]['cycles'][cycle_num]['Qd'])):
                Discharge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qd'][i] / 1.1) * 100
                if Discharge_SOC > 0.01:

                    for p in range(data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Discharge_Voltage = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                    'Voltage' : Discharge_Voltage})
                    
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)
                    Discharge_Voltage = interpolate_timeseries(data[1], inter_value)
                    Discharge_Voltage = (Discharge_Voltage-3)/(3.6-3)

                    temporal_data.append(Discharge_Voltage)


                    break
            if np.array(temporal_data).shape == (4, inter_value):
                Cycle_val.append(bat_dict[battery_key]['summary']['cycle'][int(cycle_num)])
                SOH_val.append(bat_dict[battery_key]['summary']['QD'][int(cycle_num)])
                whole_val_data.append(np.array(temporal_data))
    
    temporal_Cycle = np.array(Cycle_val)[Cycle_index:]
    max_Cycle = np.max(temporal_Cycle)
    temporal_Cycle = temporal_Cycle/max_Cycle
    temporal_Cycle.tolist()
    
    Cycle_val[Cycle_index:] = temporal_Cycle
    Cycle_index = len(Cycle_val)

whole_val_data = np.array(whole_val_data)
Cycle_val = np.array(Cycle_val)
SOH_val = np.array(SOH_val)

whole_val_data = np.reshape(whole_val_data, (whole_val_data.shape[0], 1, whole_val_data.shape[1], whole_val_data.shape[2]))

whole_val_data = torch.tensor(whole_val_data, dtype=torch.float32)

In [ ]:
minus_val_data =[]

for battery_key in test_bat_key:
    first_data = []

    for cycle_num in range(1, len(bat_dict[battery_key]['summary']['cycle'])):
        temporal_data = []
        cycle_num = str(cycle_num)

        data_length = len(bat_dict[battery_key]['cycles'][cycle_num]['Qc'])

        if cycle_num == '1':
            for i in range(11, len(bat_dict[battery_key]['cycles'][cycle_num]['Qc'])):
                Charge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qc'][i] / 1.1) * 100
                if Charge_SOC > 80.1:
                        for p in range(data_length):
                            start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                            end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                            total_time = end_time -start_time
                            if total_time > time_value:
                                final_idx = p - 1
                                break

                        time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                        Current_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx], 3)
                        Voltage_data = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]
                        Temperature_data = bat_dict[battery_key]['cycles'][cycle_num]['T'][i:i+final_idx]

                        df = pd.DataFrame({'time' : time_data,
                                   'Current' : Current_data,
                                   'Voltage' : Voltage_data})
                        
                        df = df.drop_duplicates(['time'])

                        data = np.array(df.T)

                        Current_data = interpolate_timeseries(data[1], inter_value)
                        Voltage_data = interpolate_timeseries(data[2], inter_value)
                        Temperature_data = interpolate_timeseries(Temperature_data, inter_value)

                        first_data.append(Current_data)
                        first_data.append(Voltage_data)
                        first_data.append(Temperature_data)
                        break

            for i in range(11, len(bat_dict[battery_key]['cycles'][cycle_num]['Qd'])):
                Discharge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qd'][i] / 1.1) * 100
                if Discharge_SOC > 0.01:
                    for p in range(data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Discharge_Voltage = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                   'Voltage' : Discharge_Voltage})
                
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)
                    Discharge_Voltage = interpolate_timeseries(data[1], inter_value)


                    first_data.append(Discharge_Voltage)

                    break

        elif int(cycle_num) >= 2: 
            for i in range(11, len(bat_dict[battery_key]['cycles'][cycle_num]['Qc'])):
                Charge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qc'][i] / 1.1) * 100
                if Charge_SOC > 80.1:
                        
                        for p in range(data_length):
                            start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                            end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                            total_time = end_time -start_time
                            if total_time > time_value:
                                final_idx = p - 1
                                break
                        

                        Distinguish_data = bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx]
                        has_negative = np.any(Distinguish_data < 0)
                        if has_negative:
                            break


                        time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                        Current_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['I'][i:i+final_idx],3)
                        Voltage_data = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]
                        Temperature_data = bat_dict[battery_key]['cycles'][cycle_num]['T'][i:i+final_idx]

                        df = pd.DataFrame({'time' : time_data,
                                   'Current' : Current_data,
                                   'Voltage' : Voltage_data})
                
                        df = df.drop_duplicates(['time'])

                        data = np.array(df.T)

                        Current_data = interpolate_timeseries(data[1], inter_value) - first_data[0]
                        Voltage_data = interpolate_timeseries(data[2], inter_value) - first_data[1]
                        Temperature_data = interpolate_timeseries(Temperature_data, inter_value) - first_data[2]
                    

                        temporal_data.append(Current_data)
                        temporal_data.append(Voltage_data)
                        temporal_data.append(Temperature_data)
                        break

            for i in range(11, len(bat_dict[battery_key]['cycles'][cycle_num]['Qd'])):
                Discharge_SOC = (bat_dict[battery_key]['cycles'][cycle_num]['Qd'][i] / 1.1) * 100
                if Discharge_SOC > 0.01:

                    for p in range(data_length):
                        start_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i]
                        end_time = bat_dict[battery_key]['cycles'][cycle_num]['t'][i+p]
                        total_time = end_time -start_time
                        if total_time > time_value:
                            final_idx = p - 1
                            break

                    time_data = np.round(bat_dict[battery_key]['cycles'][cycle_num]['t'][i:i+final_idx], 3)
                    Discharge_Voltage = bat_dict[battery_key]['cycles'][cycle_num]['V'][i:i+final_idx]

                    df = pd.DataFrame({'time' : time_data,
                                   'Voltage' : Discharge_Voltage})
                
                    df = df.drop_duplicates(['time'])

                    data = np.array(df.T)
                    Discharge_Voltage = interpolate_timeseries(data[1], inter_value) - first_data[3]

                    temporal_data.append(Discharge_Voltage)

                    break

            minus_val_data.append(np.array(temporal_data))

for i in range(len(minus_val_data)-1 ,0, -1):
    if (minus_val_data[i].shape != (4, inter_value)):
        del minus_val_data[i]

minus_val_data = np.array(minus_val_data)


minus_val_data = np.reshape(minus_val_data, (minus_val_data.shape[0], 1, minus_val_data.shape[1], minus_val_data.shape[2]))

minus_val_data = torch.tensor(minus_val_data, dtype=torch.float32)

In [ ]:
whole_val_data = whole_val_data.to(DEVICE)
minus_val_data = minus_val_data.to(DEVICE)

SOH_pred_val, Cycle_pred_val = model_new(whole_val_data, minus_val_data)
        
SOH_pred_val = SOH_pred_val.cpu().detach().numpy()
Cycle_pred_val = Cycle_pred_val.cpu().detach().numpy()

SOH_val_loss = mean_squared_error(SOH_val, SOH_pred_val)**0.5
Cycle_val_loss = mean_squared_error(Cycle_val, Cycle_pred_val)**0.5

SOH_mape = MAPE(np.array(SOH_val), np.array(SOH_pred_val))
Cycle_mape = MAPE(np.array(Cycle_val), np.array(Cycle_pred_val))

SOH_rmspe = RMSPE(np.array(SOH_val), np.array(SOH_pred_val))*100
Cycle_rmspe = RMSPE(np.array(Cycle_val), np.array(Cycle_pred_val))*100

print("SOH_RMSPE : ", str(np.round(SOH_rmspe, 4)), ' %')
#print("Cycle_RMSPE : ", str(np.round(Cycle_rmspe, 4)), ' %')
print("SOH_MAPE : ", str(np.round(SOH_mape, 4)), ' %')
print("Cycle_MAPE : ", str(np.round(Cycle_mape, 4)), ' %')